<a href="https://colab.research.google.com/github/PrasannaMadiwar/LLM-from-Scratch-/blob/main/Entire_GPT_2_140M_Model_fromSrcatch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import tiktoken
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class MultiHead(nn.Module):
  def __init__(self,d_in,d_out,context_length,drop_rate,n_heads,qkv_bias=False):

    super().__init__()
    assert(d_out % n_heads == 0)

    self.d_out = d_out
    self.n_heads = n_heads
    self.head_d = d_out // n_heads

    self.query = nn.Linear(d_in,d_out,bias=qkv_bias)
    self.key = nn.Linear(d_in,d_out,bias=qkv_bias)
    self.value = nn.Linear(d_in,d_out,bias=qkv_bias)
    self.dropout = nn.Dropout(drop_rate)

    self.register_buffer('mask',torch.triu(torch.ones(context_length,context_length),diagonal=1))


  def forward(self,x):
    b,n_tokens,d_in = x.shape

    keys = self.key(x)
    values = self.value(x)
    queries = self.query(x)

    keys = keys.view(b,n_tokens,self.n_heads,self.head_d)
    values = values.view(b,n_tokens,self.n_heads,self.head_d)
    queries = queries.view(b,n_tokens,self.n_heads,self.head_d)

    keys = keys.transpose(1,2)
    values = values.transpose(1,2)
    queries = queries.transpose(1,2)

    attn_score = queries @ keys.transpose(2,3)
    mask_bool = self.mask.bool()[:n_tokens,:n_tokens]
    attn_score.masked_fill(mask_bool,-torch.inf)
    attn_weights = torch.softmax(attn_score / keys.shape[-1]**0.5,dim=-1)
    attn_weights = self.dropout(attn_weights)

    context_vec = (attn_weights @ values).transpose(1,2)
    context_vec = context_vec.contiguous().view(b,n_tokens,self.d_out)
    return context_vec





In [ ]:
class LayerNorm(nn.Module):

  def __init__(self,d_in):
    super().__init__()
    self.esp = 1e-5
    self.scale = torch.ones(d_in)
    self.shift = torch.zeros(d_in)

  def forward(self,x):
    mean = x.mean(dim=-1,keepdim=True)
    var = x.var(dim=-1,keepdim=True)
    norm_x = (x - mean) / torch.sqrt(var + self.esp)

    return norm_x


In [ ]:
class Gelu(nn.Module):
  def __init__(self):
    super().__init__()
  def forward(self,x):
    return 0.5 * x * (1 + torch.tanh(x * 0.7978845608 * (1 + 0.044715 * x * x)))

In [ ]:
class FeedForward(nn.Module):
  def __init__(self,cfg):
    super().__init__()

    self.layers = nn.Sequential(
        nn.Linear(cfg['d_in'],cfg['d_in']*4),
        Gelu(),
        nn.Linear(cfg['d_in']*4,cfg['d_in'])
    )
  def forward(self,x):
    return self.layers(x)

In [ ]:
class TransFormer(nn.Module):
  def __init__(self,cfg):
    super().__init__()

    self.norm1 = LayerNorm(cfg['d_in'])
    self.norm2 = LayerNorm(cfg['d_in'])
    self.M_attn = MultiHead(cfg['d_in'],cfg['d_out'],cfg['context_length'],cfg['drop_rate'],cfg['n_heads'])
    self.ff = FeedForward(cfg)
    self.drop_out = nn.Dropout(cfg['drop_rate'])

  def forward(self,x):
    shortcut = x
    x = self.norm1(x)
    x = self.M_attn(x)
    x = self.drop_out(x)
    x = shortcut + x

    shortcut = x
    x = self.norm2(x)
    x = self.ff(x)
    x =  self.drop_out(x)
    x = shortcut + x

    return x



In [ ]:
class GPT_140M(nn.Module):
  def __init__(self,cfg):

    super().__init__()

    self.word_emb = nn.Embedding(cfg['vocab_size'],cfg['d_out'])
    self.pos_emb = nn.Embedding(cfg['context_length'],cfg['d_out'])
    self.dropout = nn.Dropout(cfg['drop_rate'])
    self.trf = nn.Sequential(
        *[TransFormer(cfg) for i in range(cfg['n_layers'])]
    )
    self.final_norm = LayerNorm(cfg['d_out'])
    self.output_head = nn.Linear(cfg['d_out'],cfg['vocab_size'],bias=False)

  def forward(self,ind_x):
    b,n_seq =  ind_x.shape
    tok_emb = self.word_emb(ind_x)
    pos_emb = self.pos_emb(torch.arange(n_seq,device=ind_x.device))
    x = tok_emb + pos_emb
    x = self.dropout(x)
    x = self.trf(x)
    x = self.final_norm(x)
    logits = self.output_head(x)
    return logits

In [ ]:
cfg = {
    'vocab_size':50257,
    'd_out':768,
    'd_in':768,
    'context_length':256,
    'drop_rate':0.1,
    'n_heads':12,
    'n_layers':12,
}

In [ ]:
model = GPT_140M(cfg)
tokenizer = tiktoken.get_encoding('gpt2')

In [ ]:
with open('/content/alice_in_wonderland.txt',encoding='utf-8') as f:
  text = f.read()


In [ ]:
len(text)

148208

In [ ]:
split_index = int(0.9*len(text))

In [ ]:
train_data = text[:split_index]
val_data = text[split_index:]

In [ ]:
val_data

' it.)\n\n  `I\'m glad I\'ve seen that done,\' thought Alice.  `I\'ve so often\nread in the newspapers, at the end of trials, "There was some\nattempts at applause, which was immediately suppressed by the\nofficers of the court," and I never understood what it meant\ntill now.\'\n\n  `If that\'s all you know about it, you may stand down,\'\ncontinued the King.\n\n  `I can\'t go no lower,\' said the Hatter:  `I\'m on the floor, as\nit is.\'\n\n  `Then you may SIT down,\' the King replied.\n\n  Here the other guinea-pig cheered, and was suppressed.\n\n  `Come, that finished the guinea-pigs!\' thought Alice.  `Now we\nshall get on better.\'\n\n  `I\'d rather finish my tea,\' said the Hatter, with an anxious\nlook at the Queen, who was reading the list of singers.\n\n  `You may go,\' said the King, and the Hatter hurriedly left the\ncourt, without even waiting to put his shoes on.\n\n  `--and just take his head off outside,\' the Queen added to one\nof the officers:  but the Hatter was out

In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []


        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})


        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):


    tokenizer = tiktoken.get_encoding("gpt2")


    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)


    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [ ]:
train_loader = create_dataloader_v1(   train_data,
    batch_size=1,
    max_length=cfg["context_length"],
    stride=cfg["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0)
val_loader = create_dataloader_v1(val_data,
    batch_size=1,
    max_length=cfg["context_length"],
    stride=cfg["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0)

In [ ]:
def encode(text, tokenizer):
    idx = tokenizer.encode(text)
    return torch.tensor(idx).unsqueeze(0)

def decode(idx, tokenizer):
     idx = idx.squeeze(0).tolist()
     text = tokenizer.decode(idx)
     return text


In [ ]:
def calc_loss_batch(input_batch,target_batch,model,device):
  input_batch,target_batch = input_batch.to(device),target_batch.to(device)
  logits = model(input_batch)
  loss = torch.nn.functional.cross_entropy(logits.flatten(0,1),target_batch.flatten())
  return loss

In [ ]:
def calc_loss_loader(data_loader,model,device,num_batches=None):
  total_loss = 0
  if len(data_loader)<0:
    return float('nan')
  elif num_batches is None:
    num_batches = len(data_loader)
  else:
    num_batches= min(num_batches,len(data_loader))
  for i, (inpuut_batch,target_batch) in enumerate(data_loader):
    if i < num_batches:
      loss = calc_loss_batch(input_batch=inpuut_batch,target_batch=target_batch,model=model,device=device)
      total_loss += loss.item()
    else:
      break
  return total_loss/num_batches


In [ ]:
model.to(device)

GPT_140M(
  (word_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (trf): Sequential(
    (0): TransFormer(
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (M_attn): MultiHead(
        (query): Linear(in_features=768, out_features=768, bias=False)
        (key): Linear(in_features=768, out_features=768, bias=False)
        (value): Linear(in_features=768, out_features=768, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): Gelu()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (drop_out): Dropout(p=0.1, inplace=False)
    )
    (1): TransFormer(
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (M_attn): MultiHead(
        (query): Linear(in_features=768, out_features=768, bias=False)
        (key): Linear(in_

In [ ]:
len(train_loader)

148

In [ ]:
len(val_loader)

16

In [ ]:
'''torch.manual_seed(123)
with torch.no_grad():
  train_loss = calc_loss_loader(train_loader,model,device,num_batches=1)
  val_loss = calc_loss_loader(val_loader,model,device,num_batches=1)
print("Training_Loss:"+str(train_loss))
print("Validation_Loss:"+str(val_loss))'''

'torch.manual_seed(123)\nwith torch.no_grad():\n  train_loss = calc_loss_loader(train_loader,model,device,num_batches=1)\n  val_loss = calc_loss_loader(val_loader,model,device,num_batches=1)\nprint("Training_Loss:"+str(train_loss))\nprint("Validation_Loss:"+str(val_loss))'

In [ ]:
from os import replace
def generate_text(model,idx,max_new,context_size,tempreture,top_k):
  model.eval()
  idx = idx.to(device)
  for i in range(max_new):
    idx_cond = idx[:, -context_size:]
    logits = model(idx_cond)
    logits = logits[:,-1,:] / tempreture
    top_log,_ = torch.topk(logits,k=top_k)
    min_v = torch.min(top_log)
    logits[logits<min_v] = -float('inf')
    prob = torch.softmax(logits,dim=-1)
    id_new = torch.multinomial(prob,num_samples=1)
    idx = torch.cat((idx,id_new),dim=1)
  text = decode(idx,tokenizer=tokenizer)
  print(text.replace("\n", " "))
  model.train()



In [ ]:
def evaluate_model(model,train_loader,val_loader,device,eval_iter):
  model.eval()
  with torch.no_grad():
    val_loss = calc_loss_loader(val_loader,model,device,num_batches=eval_iter)
    train_loss = calc_loss_loader(train_loader,model,device,num_batches=eval_iter)
    model.train()
  return train_loss,val_loss

In [ ]:
def trainGpt(model,train_loader,val_loader,optimizer,device,num_epoch,eval_iter,eval_freq,start_context,tokenizer):
  train_losses,val_losses,seen_tokens = [],[],[]
  token_seen,global_step = 0,-1

  for i in range(num_epoch):
    model.train()
    for input_batch,target_batch in train_loader:
      optimizer.zero_grad()
      loss = calc_loss_batch(input_batch,target_batch,model,device)
      loss.backward()
      optimizer.step()
      train_losses.append(loss.item())
      token_seen += input_batch.numel()
      global_step += 1
      '''if global_step % eval_freq == 0:
        train_loss,val_loss = evaluate_model(model,train_loader,val_loader,device,eval_iter)
        print("Epoch:"+str(i)+" Training_Loss:"+str(train_loss)+" Validation_Loss:"+str(val_loss))
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        seen_tokens.append(token_seen)'''

    #generate_text(model=model,idx= encode(start_context,tokenizer=tokenizer),max_new=50,context_size=256,tempreture=1.21,top_k=5)
  return train_losses,val_losses,seen_tokens

In [ ]:

torch.manual_seed(123)
model.to(device)
optimizer= torch.optim.AdamW(model.parameters(),lr=0.0004,weight_decay=0.1)
num_epoch = 10

In [ ]:
train_loss,val_loss,n_token_seen = trainGpt(model,train_loader=train_loader,val_loader=val_loader,device=device,optimizer=optimizer,num_epoch=num_epoch,eval_iter=5,eval_freq=5,start_context="hello how are t",tokenizer=tokenizer)

In [ ]:
torch.save(model.state_dict(),'model.pth')

In [ ]:
train_loss,val_loss = evaluate_model(model,train_loader,val_loader,device,4)
print("train_loss: "+str(train_loss)+" val_loss: "+str(val_loss))


train_loss: 1.0370284616947174 val_loss: 4.429116785526276


In [ ]:
start_context = "hey i am a boy i want to be an pilot"
generate_text(model=model,idx= encode(start_context,tokenizer=tokenizer),max_new=400,context_size=256,tempreture=1.21,top_k=10)

hey i am a boy i want to be an pilot out of a large an off again!' said a serpent, that's a book a very a great a queer with a few an ignorant a mouse, a down a tree a little.    a tree a mouse, a this a mouse a little girl a little golden at herself all what a sort!' cried Alice seemed to be a little."' she soon found herself a mouse a mouse' said very soon very glad at any wine out-pan a mouse a bit a very a little nervous out a queer an arrow a large She's a mouse a much out of a little another a mouse, at any a mouse:  So after a little now!' Alice a time down a shower a game.'    The Mouse only a little golden at its head a little timid a tree a little of a bottle near a much a an M off again!' pleaded get an important such a large pigeon gave a little again!'  a mouse again!' then a little golden key like a little glass:  There was a little startled off a little mouse a littleing quite pleased a mouse to a little a looking. of such a little timid a little'; a mouse again!' then k